# Cleanup: 02 — Property graph teardown

**Removes** the BigQuery property graph created by `02_gdelt_graph_create.ipynb`:
- `DROP PROPERTY GRAPH IF EXISTS ...GdeltGraph`

**IAM**: BigQuery Data Editor or roles that allow `DROP` on property graphs in the target dataset.

**Irreversible** for the graph object (you can recreate it by re-running notebook 02).

**Config:** Project and dataset identifiers come from **`config.py`** (loaded by the code cells below).

### Full reset order

1. **`02_gdelt_graph_create_cleanup.ipynb`** — Drop the BigQuery property graph while datasets still exist.
2. **`03_gdelt_incremental_refresh_cleanup.ipynb`** — Remove `_sync_metadata` and staging tables (optional if you immediately run step 4).
3. **`04_gdelt_data_profiling_cleanup.ipynb`** — Remove Dataplex scans and generated descriptions (dataset must still exist).
4. **`01_gdelt_data_prep_cleanup.ipynb`** — Delete `{BIGQUERY_DATASET}`, `{BIGQUERY_DATASET}_us`, and local exports (**always last**).

```mermaid
flowchart LR
  cleanup02[cleanup_02_drop_graph]
  cleanup03[cleanup_03_incremental_artifacts]
  cleanup04[cleanup_04_dataplex_and_descriptions]
  cleanup01[cleanup_01_datasets_and_local]
  cleanup02 --> cleanup01
  cleanup03 --> cleanup01
  cleanup04 --> cleanup01
```

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_cfg_dir = _cwd.parent if _cwd.name == "clean_up" else _cwd
if not (_cfg_dir / "config.py").is_file():
    _cfg_dir = _cwd
sys.path.insert(0, str(_cfg_dir))

from config import GCP_PROJECT_ID, BIGQUERY_DATASET, print_config, validate_config

print_config()
if not validate_config():
    print("\n⚠️  Update config.py before proceeding.")
else:
    print("\n✅ Configuration loaded.")

In [ ]:
from google.cloud import bigquery
from google.cloud.exceptions import NotFound

client = bigquery.Client(project=GCP_PROJECT_ID)
ds_ref = bigquery.DatasetReference(GCP_PROJECT_ID, BIGQUERY_DATASET)
try:
    client.get_dataset(ds_ref)
except NotFound:
    print(
        f"ℹ️  Dataset `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}` not found — "
        "property graph cleanup skipped (nothing to remove)."
    )
else:
    graph_fqn = f"`{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.GdeltGraph`"
    sql = f"DROP PROPERTY GRAPH IF EXISTS {graph_fqn}"
    print(f"Executing: {sql}")
    client.query(sql, location="US-CENTRAL1").result()
    print("✅ Property graph dropped (or did not exist).")